# G3 — OI Proxy Migration (OHLCV1D Volume)

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)

# SEAGATE required for E notebooks


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (wk, wm) in zip(axes, WINDOWS_META.items()):
    # load local volume_arc
    rk = wm['result_key']
    varc = pd.read_parquet(RESULTS_DIR / rk / 'none' / 'volume_arc.parquet')
    bs_local = varc['back_share'].values

    # load OHLCV1D from SEAGATE as proxy
    ohlcv_files = list(SEAGATE_DIR.glob(f'ohlcv1d_{wm["front"]}_{wm["back"]}*.parquet'))
    if not ohlcv_files:
        ohlcv_files = list(SEAGATE_DIR.glob(f'ohlcv*{wm["front"]}*{wm["back"]}*.parquet'))
    if ohlcv_files:
        ohlcv = pd.read_parquet(ohlcv_files[0])
        ohlcv['date'] = pd.to_datetime(ohlcv.index if ohlcv.index.name == 'ts_event'
                                        else ohlcv.get('ts_event', ohlcv.index)).dt.date.astype(str)
        days_in_roll = wm['days']
        front_vols, back_vols = [], []
        for d in days_in_roll:
            day_data = ohlcv[ohlcv['date'] == d]
            fv = day_data[day_data.get('symbol', day_data.index) == wm['front']]['volume'].sum() if 'symbol' in ohlcv.columns else np.nan
            bv = day_data[day_data.get('symbol', day_data.index) == wm['back']]['volume'].sum()  if 'symbol' in ohlcv.columns else np.nan
            front_vols.append(fv); back_vols.append(bv)
        fv_arr = np.array(front_vols, dtype=float)
        bv_arr = np.array(back_vols,  dtype=float)
        total = fv_arr + bv_arr
        bs_ohlcv = np.where(total > 0, bv_arr / total, np.nan)
        ax.plot(range(7), bs_ohlcv, ls='--', marker='s', color=WIN_COLORS[wk],
                alpha=0.6, label='OHLCV1D proxy')
        del ohlcv; gc.collect()

    ax.plot(range(len(bs_local)), bs_local, marker='o', color=WIN_COLORS[wk],
            label='Results arc', lw=1.8)
    ax.axhline(0.5, color='black', ls=':', lw=0.8)
    ax.set_xticks(range(7)); ax.set_xticklabels(['D1','D2','D3','D4','D5','D6','D7'])
    ax.set_title(wk); ax.legend(fontsize=7)
    ax.set_ylabel('Back Share')
    ax.set_xlabel('Roll Day')
    ax.annotate('* volume proxy,\nnot true OI', xy=(0.02,0.02), xycoords='axes fraction', fontsize=7)
fig.suptitle('Volume Migration: Results Arc vs OHLCV1D Proxy\n(note: OHLCV1D volume ≠ open interest)', fontsize=12)
save_fig(fig, 'G3_oi_proxy_migration.png')
